# Train Mnx: tools, images, search, code and answers

Fine-tunes **Qwen2.5-3B-Instruct** into your Mnx model on up to **100,000 conversations** that match exactly how the Mnx app talks to the model (`<think>`, `<tool_call>`, `<tool_response>`), then **measures** it on 600 held-out cases and exports GGUF files for your phone.

**Runtime → Change runtime type → GPU.** Rough training time for the full 100,000 conversations (~160M tokens, 1 epoch):

| GPU | Time (approx.) |
|---|---|
| T4 (free) | ~25–35 h → needs several sessions; progress is saved to Google Drive and resumes |
| L4 | ~10–12 h |
| A100 | ~4–6 h |

Short on time? Set `USE = 20000` below: still very diverse, ~5–7 h on a T4.

In [ ]:
# 1. Check the GPU, install Unsloth, connect Google Drive (to keep progress)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q unsloth
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/mnx-training"

## 1b. Get the Mnx code
If the repo is private, add a Colab secret `GITHUB_TOKEN` (🔑 on the left). Add `HF_TOKEN` (write access) to upload the model at the end.

In [ ]:
import os, subprocess
from google.colab import userdata
try:
    gh = userdata.get("GITHUB_TOKEN")
except Exception:
    gh = None
url = f"https://{gh}@github.com/Tharunmaks/Mnx" if gh else "https://github.com/Tharunmaks/Mnx"
if not os.path.exists("Mnx"):
    subprocess.run(["git", "clone", "-q", "-b", "claude/mnx-ai-web-interface-f75vu6", url, "Mnx"], check=True)
!cd Mnx && git log --oneline -1

## 2. Make the data
Generates 100,000 distinct conversations (~600 MB, under a minute) and 600 held-out test cases, then checks every conversation with Mnx's own parsers.

**Optional:** to keep what your old fine-tune learned, upload its data as `extra.jsonl` (one `{"messages": [...]}` per line).

In [ ]:
!cd Mnx && python3 training/generate.py --train 100000 --test 600
!cd Mnx && npm --version > /dev/null 2>&1 || (curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs > /dev/null)
!cd Mnx && npm install --omit=optional --silent && node training/validate.mjs

## 3. Train (LoRA)

In [ ]:
USE = 100000      # how many conversations to train on (e.g. 20000 for a ~6 h T4 run)
# Smaller base models answer much faster on phones but are less capable:
#   "unsloth/Qwen2.5-3B-Instruct"   best answers (default)
#   "unsloth/Qwen2.5-1.5B-Instruct" about 2x faster on a phone, trains ~2x faster
#   "unsloth/Qwen2.5-0.5B-Instruct" about 5x faster, noticeably weaker
BASE = "unsloth/Qwen2.5-3B-Instruct"
EPOCHS = 1
MAX_LEN = 4096

from unsloth import FastLanguageModel
from datasets import load_dataset, concatenate_datasets
import os

model, tokenizer = FastLanguageModel.from_pretrained(BASE, max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

data = load_dataset("json", data_files="Mnx/training/data/train.jsonl", split="train").shuffle(seed=3407).select(range(USE))
if os.path.exists("extra.jsonl"):
    extra = load_dataset("json", data_files="extra.jsonl", split="train")
    extra = extra.map(lambda ex: {"category": "extra", "train_from": max(i for i, m in enumerate(ex["messages"]) if m["role"] == "user")})
    data = concatenate_datasets([data, extra.select_columns(data.column_names)]).shuffle(seed=3407)

HEAD = tokenizer("<|im_start|>assistant\n", add_special_tokens=False)["input_ids"]
NL = tokenizer("\n", add_special_tokens=False)["input_ids"]

def to_features(ex):
    # Render exactly like Qwen's chat template, and learn ONLY the assistant
    # messages after the user's latest message (thinking, tool calls, answer).
    # Earlier turns, the system prompt and tool results are context only.
    ids, labels = [], []
    for i, m in enumerate(ex["messages"]):
        seg = tokenizer(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n", add_special_tokens=False)["input_ids"]
        if m["role"] == "assistant" and i > ex["train_from"]:
            assert seg[:len(HEAD)] == HEAD
            body = seg[len(HEAD):]
            tail = len(NL) if body[-len(NL):] == NL else 0   # don't learn the newline after <|im_end|>
            labels += [-100] * len(HEAD) + body[:len(body) - tail] + [-100] * tail
        else:
            labels += [-100] * len(seg)
        ids += seg
    return {"input_ids": ids[:MAX_LEN], "attention_mask": [1] * min(len(ids), MAX_LEN), "labels": labels[:MAX_LEN]}

data = data.map(to_features, remove_columns=data.column_names, num_proc=2)
print(len(data), "conversations;", sum(len(x) for x in data["input_ids"][:1000]) // 1000, "tokens each on average")

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported
import glob

args = TrainingArguments(
    output_dir=SAVE_DIR, per_device_train_batch_size=4, gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_steps=50, lr_scheduler_type="cosine",
    optim="adamw_8bit", weight_decay=0.01, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
    logging_steps=25, save_steps=250, save_total_limit=2, group_by_length=True, seed=3407, report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=data,
                  data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))
# Resumes automatically from the last checkpoint on Google Drive after a disconnect:
# just re-run the setup cells and this one.
trainer.train(resume_from_checkpoint=bool(glob.glob(f"{SAVE_DIR}/checkpoint-*")))
model.save_pretrained(f"{SAVE_DIR}/lora-final"); tokenizer.save_pretrained(f"{SAVE_DIR}/lora-final")

## 3b. Quick check

In [ ]:
FastLanguageModel.for_inference(model)
import json
case = json.loads(open("Mnx/training/data/test.jsonl").readline())
ids = tokenizer.apply_chat_template(case["messages"], add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=ids, max_new_tokens=300, temperature=0.7, top_p=0.8, top_k=20)
print("USER:", case["messages"][-1]["content"])
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=False))

## 4. Export GGUF for the phone
- **Q4_K_M**: best quality for its size
- **Q4_0**: recent llama.cpp builds run this faster on most phone CPUs (ARM). Try both.

In [ ]:
model.save_pretrained_gguf("mnx-gguf", tokenizer, quantization_method=["q4_k_m", "q4_0"])
import shutil
for f in glob.glob("mnx-gguf*/*.gguf") + glob.glob("mnx-gguf*.gguf"):
    low = f.lower()
    if "q4_k_m" in low: shutil.copy(f, "mnx-q4_k_m.gguf")
    elif "q4_0" in low: shutil.copy(f, "mnx-q4_0.gguf")
for f in ["mnx-q4_k_m.gguf", "mnx-q4_0.gguf"]:
    shutil.copy(f, SAVE_DIR)   # keep a copy on Drive
!ls -lh mnx-*.gguf

## 5. Measure accuracy (target 98%)
Runs 600 held-out cases through Mnx's real tool loop (including the permission step for code) with llama.cpp on the GPU. Builds llama.cpp once (~5–8 minutes).

In [ ]:
if not os.path.exists("llama.cpp/build/bin/llama-server"):
    !git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
    !cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF > /dev/null
    !cmake --build llama.cpp/build --config Release -j 4 --target llama-server > /dev/null
!cd Mnx && MNX_LLAMA_SERVER_BIN=../llama.cpp/build/bin/llama-server MNX_GPU_LAYERS=99 node training/eval.mjs --model ../mnx-q4_k_m.gguf --target 98

Below 98% somewhere? The failures are printed above and saved in `Mnx/training/data/eval-report.json`. Send them to Claude to improve the data, or train another epoch (`EPOCHS = 2`).

## 6. Upload to Hugging Face

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=userdata.get("HF_TOKEN"))
repo = "tharunmakes/mnx-qwen2.5-3b-gguf"
api.create_repo(repo, private=True, exist_ok=True)
for f in ["mnx-q4_k_m.gguf", "mnx-q4_0.gguf"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=repo)
print("Uploaded. On the phone: HF_TOKEN=hf_... npm run get-model")